# Launch, watch, and steer an Ergodis solve

This notebook starts a campaign, runs a controlled search under it, watches the search live, activates an ordering plan while the search is still running, and then queries what the run produced.

Nothing here adds instrumentation to the Ergodis core. A campaign already writes a provenance manifest and an append-only ledger flushed on every append, and a controlled search already streams coarse progress snapshots from an auxiliary watcher thread rather than from its hot loop, so watching costs the search nothing.

Launch with `analysis/ergodis-notebook` from the `ergodis-private` root: it puts the package on `PYTHONPATH` and picks a run root short enough for a Unix socket path.

In [ ]:
import os
import shutil
import time
from pathlib import Path

from ergodis_notebook import Campaign, Monitor, alignment_search, binary, open_catalog, plan
from ergodis_notebook.paths import PRIVATE_ROOT

# A controlled search binds its watcher socket inside the run directory, and
# Unix socket paths are capped at 108 bytes, so the run root has to be short.
default_root = f"{os.environ.get('XDG_RUNTIME_DIR', '/tmp')}/ergodis-runs"
run_root = Path(os.environ.get("ERGODIS_RUN_ROOT", default_root))
run_root.mkdir(parents=True, exist_ok=True)
run_dir = run_root / "solve-monitor"
shutil.rmtree(run_dir, ignore_errors=True)

data = PRIVATE_ROOT / "examples" / "data" / "campaign-c880-live-ordering.jsonl"
print(binary("ergodis-campaign"))
print(binary("alignment-controlled"))

## Start the campaign

The campaign is the control plane, not the solver. It holds the frozen feature batch that candidate plans are written against, serves the control socket, and records the ledger. An ordinary seconds-scale workflow needs none of this; a campaign is for a search you intend to watch or steer.

In [ ]:
campaign = Campaign.launch(data=data, run_dir=run_dir)
status = campaign.status()
print(campaign.manifest.problem, "|", status["health"], "|", status["rows"], "rows")
print("fields:", ", ".join(status["fields"]))
campaign.note("opened from the solve-monitor notebook")

## Start the search

`alignment_search` runs the C880 aligned-attachment search with `--run-dir` pointing at the campaign, so the search attaches to that control plane and honours plans activated while it runs. `points=8, budget=10` completes in about half a minute and emits a progress snapshot roughly once a second.

In [ ]:
solve = alignment_search(
    campaign,
    points=8,
    budget=10,
    progress_file=run_dir / "progress.jsonl",
    pulse_interval=4096,
)
print(" ".join(solve.command))

## Watch it

One updating cell: the provenance header, campaign state, the latest progress snapshot with each counter's change since the previous snapshot, and the ledger tail. Interrupting the cell stops watching, not the search.

This first watch is time-boxed so the next cell can steer the search while it is still running.

In [ ]:
monitor = Monitor(campaign, solve)
monitor.watch(interval=1.0, timeout=6.0, stop_when_finished=False)

## Steer it, without restarting it

A running search is steered through the campaign, never through the process. Activating a plan bumps the campaign's epoch; the search notices at its next safe point and swaps in a preallocated arena, so the change costs no allocation in the search loop.

Two things are worth knowing before writing a plan.

The socket accepts the **bytecode** plan form only, not the readable expression form, so `plan.plan(...)` lowers the expression here before sending it. And a plan's `scope` is a membership bitset indexed by the field's value, not a bitwise AND mask: scoping to `root_orbit == 6` sets bit 6, giving mask 64. Passing the values themselves to `plan.plan(scope=...)` builds the mask correctly.

An `ordering` plan changes what the search explores first. It cannot change what is admitted, so the exact answer is unaffected by construction.

In [ ]:
ordering = plan.plan(
    "notebook-child-unresolved",
    plan.field("child_unresolved_count"),
    role="ordering",
    output="score",
)
print("program:", ordering["program"])
print("epoch before:", campaign.epoch())
applied = campaign.candidate_apply(ordering)
campaign.note("activated the notebook ordering plan mid-run")
print("epoch after: ", campaign.epoch())
print("score range: ", applied["evaluation"]["minimum_score"], "to", applied["evaluation"]["maximum_score"])

In [ ]:
monitor.watch(interval=1.0)

## What the search did

The monitor keeps every snapshot it collected, so the run is analyzable without re-reading anything. `notified_epoch` rising from 0 to 1 partway through is the search acknowledging the plan activated above.

`states` is the exact search-table occupancy. Its slope is the useful live signal, because the table is pre-sized and never grows: a search that outgrows it exits with "the pre-sized search table is full" rather than degrading, and `seen_capacity` is the knob.

In [ ]:
frame = monitor.progress_frame()
print(frame.shape[0], "snapshots; epochs seen:", sorted(set(frame["notified_epoch"])))
frame.tail(3)[["elapsed_ms", "notified_epoch", "solver.states", "solver.duplicates", "solver.infeasible"]]

In [ ]:
axis = frame.plot(
    x="elapsed_ms",
    y=["solver.states", "solver.duplicates", "solver.infeasible"],
    figsize=(9, 4),
)
steered = frame.loc[frame["notified_epoch"] > 0, "elapsed_ms"]
if len(steered):
    axis.axvline(steered.iloc[0], color="0.4", linestyle="--")
    axis.annotate("ordering plan activated", (steered.iloc[0], 0), xytext=(6, 6),
                  textcoords="offset points", fontsize=9, color="0.3")
axis.set_xlabel("elapsed (ms)")
axis.set_ylabel("count")
axis.figure.tight_layout()

In [ ]:
solve.wait(timeout=120)
result = solve.result()
print("answer: ", result["answer"])
print("metrics:", result["metrics"])
print("control:", result["control"])

## The run, and the rest of the evidence, as SQL

The same DuckDB catalog the `analysis/ergodis-sql` shell loads, from the same SQL file, so the notebook and the shell cannot drift apart in what a paired ratio means.

`run_ledger` reads this run's ledger; the notes written above are interleaved with the machine events, so the timeline records what the search did and what a person observed while it ran. `ab_summary` is the paired A/B table over every interleaved benchmark in `evidence/`, where a `time_geomean` below one means `arm_b` was the faster arm.

In [ ]:
catalog = open_catalog()
catalog.execute("select * from run_ledger(?)", [str(run_dir)]).df()

In [ ]:
catalog.execute(
    """
    select source_file, ord, arm_a, arm_b, rounds,
           round(time_geomean, 4) as time_geomean,
           round(t, 2) as t
    from ab_summary
    order by time_geomean
    limit 10
    """
).df()

In [ ]:
campaign.shutdown()
print("campaign running:", campaign.alive())